# 06 — Index D (The Ghost)

**Objetivo.** Predecir Index_D explotando la pista del Ghost: D replica otro índice con un lag fijo.

**Pistas del enunciado.** "The Ghost" — sigue una señal oculta en OTRO índice. Esto NO es un problema de modelado, es un problema de detective.

**Nivel de esfuerzo.** BAJO-MEDIO. Una vez identificado el lag en `00_carga_y_EDA`, la predicción es casi trivial. NO entrenar NN.

**Estrategia.** Replicar `GHOST_SOURCE` desplazado `GHOST_LAG` días. Los primeros `GHOST_LAG` días del período de predicción se pueden obtener directamente de los precios reales conocidos del índice fuente (ya están en train). Los días restantes necesitan una estimación del índice fuente — se usa el baseline drift del fuente como proxy.

**Qué esperamos.** RMSE muy inferior a cualquier NN si el lag es estable en el tiempo.

**Qué NO hace.** No entrena NN. No usa datos auxiliares.

**Inputs.** `data/train_indices.csv`, `results/baselines.json`

**Output esperado.** `results/index_D.json` (approach_type = `ghost`, sin model_path).

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

OWNER = "oscar"   # <-- cada miembro pone su nombre aquí
IDX   = "Index_D"

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils import (
    load_hackathon_data, lagged_correlation,
    make_temporal_split, baseline_drift,
    backtest_autoregressive, plot_rollout,
    DATA_DIR, VAL_DAYS, V_IN_SHARED
)

os.makedirs('results', exist_ok=True)

## 1. Carga + referencia baselines

In [ ]:
data  = load_hackathon_data(DATA_DIR)
idx   = data['train_indices']
serie = idx[IDX].dropna().values

with open('results/baselines.json') as f:
    baselines = json.load(f)
print(f'Baselines para {IDX}:')
for bl, rmse in baselines[IDX].items():
    print(f'  {bl:<15} RMSE = {rmse:,.0f}')

## 2. Resultado del detective — traer de 00_carga_y_EDA

Rellenar `GHOST_SOURCE` y `GHOST_LAG` con los valores encontrados en `00_carga_y_EDA`. Si no se ha ejecutado ese notebook, reproducir aquí el análisis de `lagged_correlation`.

In [ ]:
# ⚠️ RELLENAR con los hallazgos de 00_carga_y_EDA
GHOST_SOURCE = 'Index_?'   # índice fuente
GHOST_LAG    = 0           # lag en días

# Si no está claro, reproducir el análisis aquí:
log_idx  = np.log(idx).diff().dropna()
df_corr  = lagged_correlation(log_idx, target_col='Index_D', max_lag=30)
print('Lag óptimo por candidato:')
print(df_corr.abs().idxmax())  # lag donde la correlación con D es máxima

## 3. Confirmación visual del lag

In [ ]:
if GHOST_SOURCE in idx.columns and GHOST_LAG > 0:
    fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
    axes[0].plot(log_idx['Index_D'].values,                   lw=0.8, label='Index_D ret')
    axes[0].plot(log_idx[GHOST_SOURCE].shift(GHOST_LAG).values, lw=0.8, ls='--',
                 label=f'{GHOST_SOURCE} ret (lag={GHOST_LAG})')
    axes[0].legend()
    axes[0].set_title('Log-retornos: D vs fuente desplazada')
    axes[1].plot(idx['Index_D'].values,                   lw=0.8, label='Index_D precio')
    axes[1].plot(idx[GHOST_SOURCE].shift(GHOST_LAG).values, lw=0.8, ls='--',
                 label=f'{GHOST_SOURCE} precio (lag={GHOST_LAG})')
    axes[1].legend()
    axes[1].set_title('Precios: D vs fuente desplazada')
    plt.tight_layout()
    plt.show()
else:
    print('[Rellenar GHOST_SOURCE y GHOST_LAG antes de ejecutar esta celda]')

## 4. Función de predicción Ghost

Para el backtest autorregresivo necesitamos una `predict_fn(x)` con la firma estándar. La lógica ghost opera sobre la serie del índice fuente, no sobre la ventana `x` de D.

**Estrategia para el rollout:**
- Los primeros `GHOST_LAG` días de la predicción de D son los últimos `GHOST_LAG` precios conocidos del índice fuente (ya en train — no es leakage).
- Para los días restantes (252 - GHOST_LAG), necesitamos predecir el índice fuente. Se usa `baseline_drift` del fuente como proxy.

In [ ]:
# Construir la predicción Ghost de 252 días para el período de validación
# Esta función NO sigue la firma predict_fn(x) estándar — se usa directamente.

def ghost_predict_252(serie_d, serie_fuente, lag, val_days):
    """
    Predice val_days días de serie_d usando serie_fuente con lag.
    serie_d      : precios completos de D (train + val)
    serie_fuente : precios completos del índice fuente (train + val)
    """
    # Período de train: todo excepto los últimos val_days
    fuente_train = serie_fuente[:-val_days]

    # Para los primeros `lag` días: usar los últimos `lag` precios conocidos del fuente
    # (son datos de train — no hay leakage)
    preds_ghost = list(fuente_train[-lag:]) if lag > 0 else []

    # Para los días restantes: predecir el fuente con baseline_drift
    n_restantes = val_days - lag
    if n_restantes > 0:
        preds_fuente = baseline_drift(fuente_train, n_restantes)
        preds_ghost.extend(preds_fuente)

    return np.array(preds_ghost[:val_days])


if GHOST_SOURCE in idx.columns:
    serie_fuente = idx[GHOST_SOURCE].dropna().values
    preds_ghost  = ghost_predict_252(serie, serie_fuente, GHOST_LAG, VAL_DAYS)
    print(f'preds_ghost shape: {preds_ghost.shape}  (esperado: ({VAL_DAYS},))')
else:
    print('[Rellenar GHOST_SOURCE antes de ejecutar]')

## 5. RMSE del Ghost vs baselines

In [ ]:
from utils import rmse_per_index

val_prices = serie[-VAL_DAYS:]
rmse_ghost = float(rmse_per_index(val_prices, preds_ghost))
best_bl_rmse = min(baselines[IDX].values())

print(f'Ghost RMSE backtest = {rmse_ghost:,.0f}')
print(f'Mejor baseline RMSE = {best_bl_rmse:,.0f}')
print(f'Mejora Ghost vs baseline: {(best_bl_rmse - rmse_ghost)/best_bl_rmse*100:+.1f}%')

plot_rollout(serie, preds_ghost, index_name=f'{IDX} — Ghost({GHOST_SOURCE}, lag={GHOST_LAG})',
             val_days=VAL_DAYS)

## 6. Decisión final y guardado

In [ ]:
best_bl_name = min(baselines[IDX], key=baselines[IDX].get)
_tipo_map = {'flat': 'baseline_flat', 'drift': 'baseline_drift', 'random_walk': 'baseline_rw'}

if rmse_ghost < best_bl_rmse and GHOST_SOURCE != 'Index_?':
    approach   = 'ghost'
    final_rmse = rmse_ghost
    notes      = f'Ghost: {GHOST_SOURCE} con lag={GHOST_LAG}'
else:
    approach   = _tipo_map[best_bl_name]
    final_rmse = best_bl_rmse
    notes      = f'Ghost no identificado o no mejora — usando baseline {best_bl_name}'

info = {
    'index':              IDX,
    'owner':              OWNER,
    'approach_type':      approach,
    'strategy':           approach,
    'rmse_backtest_252d': final_rmse,
    'model_path':         None,
    'log_ret_mode':       False,
    'v_in':               None,
    'ghost_source_index': GHOST_SOURCE if approach == 'ghost' else None,
    'ghost_lag':          GHOST_LAG    if approach == 'ghost' else None,
    'notes':              notes
}

with open('results/index_D.json', 'w') as f:
    json.dump(info, f, indent=2)

print('Guardado: results/index_D.json')
print(json.dumps(info, indent=2))